# Coco Crepe — 02 Silver Product Master

Limpieza, estandarización y deduplicación de productos.

In [0]:
GROUP = "g203"

SALES_DATA_PRODUCT = "sales_summary"
INVENTORY_DATA_PRODUCT = "inventory_status"
PRODUCT_DATA_PRODUCT = "product_master"

# Completa estos valores solo si la detección automática no encuentra
# exactamente un catálogo por Data Product.
SALES_CATALOG_MANUAL = None
INVENTORY_CATALOG_MANUAL = "g203_inv_inventory_status"
PRODUCT_CATALOG_MANUAL = None

def resolve_catalog(data_product_name, manual_catalog=None):
    if manual_catalog:
        return manual_catalog

    catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
    target = data_product_name.lower()

    preferred = [
        catalog for catalog in catalogs
        if GROUP.lower() in catalog.lower()
        and target in catalog.lower()
    ]

    if len(preferred) == 1:
        return preferred[0]

    matches = [
        catalog for catalog in catalogs
        if target in catalog.lower()
    ]

    if len(matches) == 1:
        return matches[0]

    raise ValueError(
        f"No se pudo identificar un catálogo único para '{data_product_name}'. "
        f"Catálogos visibles: {catalogs}. "
        "Completa la variable *_CATALOG_MANUAL correspondiente."
    )

SALES_CATALOG = resolve_catalog(
    SALES_DATA_PRODUCT,
    SALES_CATALOG_MANUAL
)

INVENTORY_CATALOG = resolve_catalog(
    INVENTORY_DATA_PRODUCT,
    INVENTORY_CATALOG_MANUAL
)

PRODUCT_CATALOG = resolve_catalog(
    PRODUCT_DATA_PRODUCT,
    PRODUCT_CATALOG_MANUAL
)

print(f"Sales catalog: {SALES_CATALOG}")
print(f"Inventory catalog: {INVENTORY_CATALOG}")
print(f"Product catalog: {PRODUCT_CATALOG}")

In [0]:
product_bronze = f"{PRODUCT_CATALOG}.bronze.products"
products_clean = f"{PRODUCT_CATALOG}.silver.products_clean" 

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {products_clean} AS
WITH standardized AS (
    SELECT
        CAST(product_id AS INT) AS product_id,
        INITCAP(TRIM(product_name)) AS product_name,
        UPPER(TRIM(category)) AS category,
        ROUND(CAST(price AS DOUBLE), 2) AS price,
        inserted_at,
        ROW_NUMBER() OVER (
            PARTITION BY product_id
            ORDER BY inserted_at DESC
        ) AS row_number
    FROM {product_bronze}
    WHERE product_id IS NOT NULL
      AND product_name IS NOT NULL
      AND TRIM(product_name) <> ''
      AND price > 0
)
SELECT
    product_id,
    product_name,
    category,
    price,
    current_timestamp() AS transformed_at
FROM standardized
WHERE row_number = 1
""")

In [0]:
spark.sql(f"""
SELECT
    COUNT(*) AS record_count,
    COUNT(DISTINCT product_id) AS distinct_products
FROM {products_clean}
""").display()

spark.sql(
    f"SELECT * FROM {products_clean} ORDER BY product_id LIMIT 20"
).display()